In [69]:
import os

import mlflow
import requests
import json
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://localhost:5000")
client = MlflowClient()

SLACK_WEBHOOK_URL = os.getenv("SLACK_WEBHOOK_URL")
REQUIRED_TAGS = ["team", "use_case", "data_version"]
# REQUIRED_ARTIFACTS = ["model", "requirements.txt"]
REQUIRED_ARTIFACTS = ["model", "environment.yml"]

### Get the Model Version Pending Review

In [70]:
MODEL_NAME = "employee-attrition"
VERSION = "6"  # version to test

mv = client.get_model_version(MODEL_NAME, VERSION)
run_id = mv.run_id

print(f"Model     : {MODEL_NAME}")
print(f"Version   : {VERSION}")
print(f"Run ID    : {run_id}")
print(f"Stage     : {mv.current_stage}")
print(f"Description: {mv.description}")

Model     : employee-attrition
Version   : 6
Run ID    : 6435f814834a4eadb2f86f0002089e21
Stage     : Archived
Description: Rejected during testing review.


In [68]:
artifacts = client.list_artifacts(run_id)

for a in artifacts:
    print(a.path)
# print("\nArtifacts:" , artifacts)

environment.yml


In [59]:
model_info = mlflow.models.get_model_info(f"models:/employee-attrition/4")
print(model_info.signature)

inputs: 
  ['employee_id': long (required), 'age': long (required), 'gender': integer (required), 'years_at_company': long (required), 'job_role': integer (required), 'monthly_income': long (required), 'work-life_balance': integer (required), 'job_satisfaction': integer (required), 'performance_rating': integer (required), 'number_of_promotions': long (required), 'overtime': integer (required), 'distance_from_home': long (required), 'education_level': integer (required), 'marital_status': integer (required), 'number_of_dependents': long (required), 'job_level': integer (required), 'company_size': integer (required), 'company_tenure': long (required), 'remote_work': integer (required), 'leadership_opportunities': integer (required), 'innovation_opportunities': integer (required), 'company_reputation': integer (required), 'employee_recognition': integer (required)]
outputs: 
  [Tensor('int32', (-1,))]
params: 
  None



### Test 1: Model Schema Accuracy

In [60]:
model_uri = f"models:/{MODEL_NAME}/{VERSION}"
model_info = mlflow.models.get_model_info(model_uri)

if model_info.signature is None:
    schema_status = "FAILED"
    schema_msg = "No model signature found. Log with mlflow.log_model(signature=...)"
else:
    inputs = model_info.signature.inputs.to_dict() if model_info.signature.inputs else []
    outputs = model_info.signature.outputs.to_dict() if model_info.signature.outputs else []

    if not inputs:
        schema_status = "FAILED"
        schema_msg = "Input schema is empty"
    elif not outputs:
        schema_status = "FAILED"
        schema_msg = "Output schema is empty"
    else:
        schema_status = "PASSED"
        schema_msg = f"{len(inputs)} input(s), {len(outputs)} output(s) found"

print(f"[{schema_status}] Model Schema Accuracy — {schema_msg}")

[PASSED] Model Schema Accuracy — 23 input(s), 1 output(s) found


### Test 2: Docs & Artifacts

In [61]:
if not mv.description:
    artifacts_status = "FAILED"
    artifacts_msg = "Model version has no description"
else:
    all_artifacts = [a.path for a in client.list_artifacts(run_id)]
    missing = [a for a in REQUIRED_ARTIFACTS if a not in all_artifacts]

    if missing:
        artifacts_status = "FAILED"
        artifacts_msg = f"Missing artifacts: {missing}"
    else:
        artifacts_status = "PASSED"
        artifacts_msg = f"All required artifacts present: {all_artifacts}"

print(f"[{artifacts_status}] Docs & Artifacts — {artifacts_msg}")

[FAILED] Docs & Artifacts — Missing artifacts: ['model']


### Test 3: Set Tags

In [62]:
existing_tags = set(mv.tags.keys()) if mv.tags else set()
missing_tags = [t for t in REQUIRED_TAGS if t not in existing_tags]


if missing_tags:
    tags_status = "FAILED"
    tags_msg = f"Missing required tags: {missing_tags}"
else:
    tags_status = "PASSED"
    tags_msg = f"All required tags present: {list(existing_tags)}"

# Stamp the model version as under review regardless
client.set_model_version_tag(MODEL_NAME, VERSION, "testing_status", "in_review")

print(f"[{tags_status}] Set Tags — {tags_msg}")
print("Tag 'testing_status=in_review' applied")

[PASSED] Set Tags — All required tags present: ['use_case', 'testing_status', 'team', 'data_version']
Tag 'testing_status=in_review' applied


### Summary & Slack Notification

In [67]:
results = {
    "Model Schema Accuracy": schema_status,
    "Docs & Artifacts":      artifacts_status,
    "Set Tags":              tags_status,
}

all_passed = all(s == "PASSED" for s in results.values())

icons = {"PASSED": "✅", "FAILED": "❌"}
lines = [
    f"*MLflow Testing Pipeline*",
    f"Model: `{MODEL_NAME}` v{VERSION}  |  Run: `{run_id}`\n",
    "*Automated Test Results:*",
]
for name, status in results.items():
    lines.append(f"{icons[status]} *{name}*: {status}")

lines.append(f"\n{'✅ All tests passed — awaiting approval' if all_passed else '❌ Some tests failed — review required'}")

slack_payload = {"text": "\n".join(lines)}
response = requests.post(SLACK_WEBHOOK_URL, json=slack_payload)
print("Slack notification sent:", response.status_code)
print("\n--- Summary ---")
for name, status in results.items():
    print(f"  {icons[status]} {name}: {status}")
print(f"\nOverall: {'PASSED' if all_passed else 'FAILED'}")

Slack notification sent: 200

--- Summary ---
  ✅ Model Schema Accuracy: PASSED
  ❌ Docs & Artifacts: FAILED
  ✅ Set Tags: PASSED

Overall: FAILED


### Human Decision: Approve → Staging

In [64]:
# Run this cell if you APPROVE the model

client.transition_model_version_stage(
    name=MODEL_NAME,
    version=VERSION,
    stage="Staging",
    archive_existing_versions=True   # archives any existing Staging version
)

client.update_model_version(MODEL_NAME, VERSION, description="Approved and promoted to Staging.")
client.set_model_version_tag(MODEL_NAME, VERSION, "testing_status", "staging")

requests.post(SLACK_WEBHOOK_URL, json={
    "text": f"✅ *APPROVED* — `{MODEL_NAME}` v{VERSION} transitioned to *Staging*"
})

print(f"✅ {MODEL_NAME} v{VERSION} → Staging")

C:\Users\aarun\AppData\Local\Temp\ipykernel_8280\3121516048.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


✅ employee-attrition v6 → Staging


###  Human Decision: Reject → Archived

In [65]:
# Run this cell if you REJECT the model

client.transition_model_version_stage(
    name=MODEL_NAME,
    version=VERSION,
    stage="Archived"
)

client.update_model_version(MODEL_NAME, VERSION, description="Rejected during testing review.")
client.set_model_version_tag(MODEL_NAME, VERSION, "testing_status", "archived")

requests.post(SLACK_WEBHOOK_URL, json={
    "text": f"❌ *REJECTED* — `{MODEL_NAME}` v{VERSION} transitioned to *Archived*"
})

print(f"❌ {MODEL_NAME} v{VERSION} → Archived")

C:\Users\aarun\AppData\Local\Temp\ipykernel_8280\331398823.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


❌ employee-attrition v6 → Archived


In [66]:
# Run this cell if you REJECT the model

client.transition_model_version_stage(
    name=MODEL_NAME,
    version=VERSION,
    stage="Archived"
)

client.update_model_version(MODEL_NAME, VERSION, description="Rejected during testing review.")
client.set_model_version_tag(MODEL_NAME, VERSION, "testing_status", "archived")

requests.post(SLACK_WEBHOOK_URL, json={
    "text": f"❌ *REJECTED* — `{MODEL_NAME}` v{VERSION} transitioned to *Archived*"
})

print(f"❌ {MODEL_NAME} v{VERSION} → Archived")

C:\Users\aarun\AppData\Local\Temp\ipykernel_8280\331398823.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


❌ employee-attrition v6 → Archived
